# Statistical Power and Sample Size

## Learning Objectives
1. Compute statistical power for given alpha, n, and effect size from first principles
2. Build a sample size calculator implementing the two-sample formula n = (z_alpha/2 + z_beta)^2 * 2*sigma^2 / delta^2
3. Visualize the four-way relationship among alpha, power, effect size, and n
4. Simulate underpowered experiments to show the false discovery inflation caused by insufficient sample size

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from typing import Optional

np.random.seed(42)
print("scipy version:", __import__('scipy').__version__)
print("Setup complete.")

## Level 1: Computing Power and Plotting Power vs n

Power = P(reject H0 | H1 is true) = P(|Z| > z_alpha/2 | delta != 0)

For a two-sample z-test with equal group sizes n:
- Non-centrality parameter: lambda = delta / (sigma * sqrt(2/n))
- Power = P(Z > z_alpha/2 - lambda) + P(Z < -z_alpha/2 - lambda)

We compute power as a function of n and plot the power curve.

In [ ]:
# -----------------------------------------------------------------------
# Level 1: Power calculation and power vs n curve
# -----------------------------------------------------------------------

def compute_power(
    n: int,
    delta: float,
    sigma: float,
    alpha: float = 0.05,
    two_tailed: bool = True
) -> float:
    """
    Compute statistical power for a two-sample z-test.

    Parameters
    ----------
    n          : sample size per group
    delta      : true difference in means (effect size in raw units)
    sigma      : assumed common standard deviation
    alpha      : Type I error rate
    two_tailed : if True, use two-tailed test

    Returns
    -------
    power : probability of rejecting H0 given H1 is true
    """
    # Non-centrality parameter: how many SEs is the true effect from 0?
    se = sigma * np.sqrt(2 / n)  # SE of difference for two equal groups
    ncp = delta / se             # non-centrality parameter (lambda)

    if two_tailed:
        z_crit = stats.norm.ppf(1 - alpha / 2)
        # Power = P(Z > z_crit - ncp) + P(Z < -z_crit - ncp)
        power = 1 - stats.norm.cdf(z_crit - ncp) + stats.norm.cdf(-z_crit - ncp)
    else:
        z_crit = stats.norm.ppf(1 - alpha)
        power = 1 - stats.norm.cdf(z_crit - ncp)

    return power


def sample_size_for_power(
    delta: float,
    sigma: float,
    target_power: float = 0.80,
    alpha: float = 0.05,
    two_tailed: bool = True
) -> int:
    """
    Compute required sample size per group to achieve target power.

    Formula: n = (z_alpha/2 + z_beta)^2 * 2 * sigma^2 / delta^2

    Parameters
    ----------
    delta        : minimum detectable effect (raw units)
    sigma        : assumed standard deviation
    target_power : desired power (e.g. 0.80)
    alpha        : Type I error rate

    Returns
    -------
    n : required sample size per group (rounded up)
    """
    beta = 1 - target_power
    z_alpha = stats.norm.ppf(1 - alpha / 2) if two_tailed else stats.norm.ppf(1 - alpha)
    z_beta = stats.norm.ppf(target_power)   # ppf(1 - beta)
    n = (z_alpha + z_beta) ** 2 * 2 * sigma ** 2 / delta ** 2
    return int(np.ceil(n))


# ------ Power vs n for a medium effect (d=0.5, sigma=10, delta=5) ------
sigma = 10.0
delta = 5.0   # Cohen's d = 0.5 (medium effect)
alpha = 0.05

ns = np.arange(5, 301, 5)
powers_two = [compute_power(n, delta, sigma, alpha, two_tailed=True) for n in ns]
powers_one = [compute_power(n, delta, sigma, alpha, two_tailed=False) for n in ns]

# Find required n for 80% and 90% power
n_80 = sample_size_for_power(delta, sigma, target_power=0.80)
n_90 = sample_size_for_power(delta, sigma, target_power=0.90)

print(f"Effect: delta={delta}, sigma={sigma}, Cohen's d = {delta/sigma:.2f}")
print(f"  Required n per group for 80% power: {n_80}")
print(f"  Required n per group for 90% power: {n_90}")
print(f"  Power at n=20:  {compute_power(20,  delta, sigma):.3f}")
print(f"  Power at n=64:  {compute_power(64,  delta, sigma):.3f}")
print(f"  Power at n=100: {compute_power(100, delta, sigma):.3f}")

# ------ Power vs n across effect sizes ------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(ns, powers_two, "steelblue", linewidth=2, label="Two-tailed (alpha=0.05)")
ax1.plot(ns, powers_one, "darkorange", linewidth=2, linestyle="--",
         label="One-tailed (alpha=0.05)")
ax1.axhline(0.80, color="red", linestyle=":", linewidth=1.5, label="80% power target")
ax1.axhline(0.90, color="green", linestyle=":", linewidth=1.5, label="90% power target")
ax1.axvline(n_80, color="red", linestyle=":", alpha=0.5)
ax1.axvline(n_90, color="green", linestyle=":", alpha=0.5)
ax1.annotate(f"n={n_80}", xy=(n_80, 0.05), fontsize=9, color="red", ha="center")
ax1.annotate(f"n={n_90}", xy=(n_90, 0.05), fontsize=9, color="green", ha="center")
ax1.set_xlabel("Sample Size per Group", fontsize=12)
ax1.set_ylabel("Power", fontsize=12)
ax1.set_title(f"Power vs n (delta={delta}, sigma={sigma}, d=0.5)", fontsize=13, fontweight="bold")
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Multiple effect sizes
effect_sizes_d = [0.2, 0.5, 0.8, 1.0]
colors_eff = ["steelblue", "darkorange", "seagreen", "crimson"]
for d, color in zip(effect_sizes_d, colors_eff):
    delta_d = d * sigma
    pows = [compute_power(n, delta_d, sigma) for n in ns]
    ax2.plot(ns, pows, color=color, linewidth=2, label=f"d={d}")

ax2.axhline(0.80, color="black", linestyle=":", linewidth=1.5, label="80% power")
ax2.set_xlabel("Sample Size per Group", fontsize=12)
ax2.set_ylabel("Power", fontsize=12)
ax2.set_title("Power vs n for Different Effect Sizes", fontsize=13, fontweight="bold")
ax2.set_ylim(0, 1.05)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("power_vs_n.png", dpi=100, bbox_inches="tight")
plt.show()

## Level 2: Sample Size Calculator and Four-Way Relationship

The four parameters (alpha, power, effect size, n) are tightly coupled.
Fixing any three determines the fourth.
We build a complete calculator and explore each trade-off.

In [ ]:
# -----------------------------------------------------------------------
# Level 2: Full sample size calculator with one-tailed vs two-tailed,
#          four-way parameter table, and alpha/beta trade-off analysis
# -----------------------------------------------------------------------

def required_n_table(
    effect_sizes_d: list,
    powers: list,
    sigma: float = 1.0,
    alpha: float = 0.05
) -> np.ndarray:
    """Build a matrix of required n values indexed by effect size and power."""
    table = np.zeros((len(effect_sizes_d), len(powers)), dtype=int)
    for i, d in enumerate(effect_sizes_d):
        delta = d * sigma
        for j, pw in enumerate(powers):
            table[i, j] = sample_size_for_power(delta, sigma, target_power=pw, alpha=alpha)
    return table


# ------ Sample size vs effect size table ------
effect_sizes = [0.2, 0.5, 0.8, 1.0]
power_levels = [0.70, 0.80, 0.90, 0.95]

n_table = required_n_table(effect_sizes, power_levels)
print("Required n per group (alpha=0.05, two-tailed)")
print(f"{'Effect d':<12}", end="")
for pw in power_levels:
    print(f" power={pw:.0%}", end="")
print()
print("-" * 60)
for i, d in enumerate(effect_sizes):
    label = {0.2: "small", 0.5: "medium", 0.8: "large", 1.0: "v.large"}.get(d, "")
    print(f"d={d:<6} ({label:<7})", end="")
    for j in range(len(power_levels)):
        print(f"  {n_table[i, j]:>7d}", end="")
    print()

# ------ One-tailed vs two-tailed comparison ------
print("
One-tailed vs Two-tailed n requirement (power=0.80, alpha=0.05):")
print(f"{'Effect d':<10}  {'One-tailed':>12}  {'Two-tailed':>12}  {'Saving':>10}")
for d in effect_sizes:
    delta = d * 1.0
    n_two = sample_size_for_power(delta, 1.0, target_power=0.80, alpha=0.05, two_tailed=True)
    n_one = sample_size_for_power(delta, 1.0, target_power=0.80, alpha=0.05, two_tailed=False)
    saving_pct = (n_two - n_one) / n_two * 100
    print(f"d={d:<6}    {n_one:>12d}  {n_two:>12d}  {saving_pct:>8.1f}%")

# ------ Impact of changing alpha on required n ------
print("
Effect of alpha on required n (d=0.5, power=0.80):")
print(f"{'Alpha':>8}  {'Required n':>12}  {'vs alpha=0.05':>14}")
n_baseline = sample_size_for_power(0.5, 1.0, target_power=0.80, alpha=0.05)
for alpha_val in [0.10, 0.05, 0.01, 0.001]:
    n_req = sample_size_for_power(0.5, 1.0, target_power=0.80, alpha=alpha_val)
    ratio = n_req / n_baseline
    print(f"{alpha_val:>8.3f}  {n_req:>12d}  {ratio:>13.2f}x")

# ------ Impact of changing power on required n ------
print("
Effect of target power on required n (d=0.5, alpha=0.05):")
print(f"{'Power':>8}  {'Required n':>12}  {'vs power=0.80':>15}")
for pw in [0.60, 0.70, 0.80, 0.90, 0.95, 0.99]:
    n_req = sample_size_for_power(0.5, 1.0, target_power=pw, alpha=0.05)
    ratio = n_req / n_baseline
    print(f"{pw:>8.2f}  {n_req:>12d}  {ratio:>14.2f}x")

## Real-World Example 1: A/B Test Design for Conversion Rate

Scenario: Baseline conversion 5%, want to detect MDE = 1 percentage point (absolute).
Compute required users per variant using Cohen's h for proportions.

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 1: A/B test sample size for conversion rate
# -----------------------------------------------------------------------
import math

def cohens_h(p1: float, p2: float) -> float:
    """Cohen's h effect size for two proportions.
    h = 2 * arcsin(sqrt(p1)) - 2 * arcsin(sqrt(p2))
    """
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))


def sample_size_for_proportions(
    p_control: float,
    p_treatment: float,
    alpha: float = 0.05,
    power: float = 0.80,
    two_tailed: bool = True
) -> int:
    """
    Sample size per group for comparing two proportions.
    Uses Cohen's h effect size and the two-sample z-test formula.
    """
    h = abs(cohens_h(p_treatment, p_control))
    z_alpha = stats.norm.ppf(1 - alpha / 2) if two_tailed else stats.norm.ppf(1 - alpha)
    z_beta = stats.norm.ppf(power)
    n = (z_alpha + z_beta) ** 2 / h ** 2
    return int(np.ceil(n))


def daily_users_needed(n_per_variant: int, daily_traffic: int) -> float:
    """Return number of days needed to run the experiment."""
    total = 2 * n_per_variant
    return total / daily_traffic


# Product A/B test parameters
p0 = 0.05   # baseline conversion rate
mde = 0.01  # minimum detectable effect (1 percentage point absolute)
p1 = p0 + mde  # 6% target

h = cohens_h(p1, p0)
print(f"Baseline: {p0:.0%}, Target: {p1:.0%}, MDE: {mde:.0%}")
print(f"Cohen's h effect size: {h:.4f}")

n_per_variant = sample_size_for_proportions(p0, p1, alpha=0.05, power=0.80)
print(f"
Required users per variant (80% power): {n_per_variant:,}")
print(f"Total users required: {2 * n_per_variant:,}")

# Feasibility analysis
for daily in [1000, 5000, 10000, 50000]:
    days = daily_users_needed(n_per_variant, daily)
    status = "OK" if days < 28 else "TOO LONG"
    print(f"  Daily traffic {daily:>7,}: {days:.1f} days [{status}]")

# MDE sensitivity: what MDE is feasible for different n
print("
Feasible MDE for common traffic scenarios (80% power, alpha=0.05):")
print(f"{'Daily users':>13}  {'Days=14':>10}  {'Feasible MDE':>15}")
for daily in [2000, 10000, 50000, 200000]:
    n_available = daily * 14 // 2   # 14-day experiment, 2 variants
    # Find MDE by binary search
    for mde_try in np.arange(0.001, 0.20, 0.001):
        p1_try = p0 + mde_try
        n_req = sample_size_for_proportions(p0, p1_try)
        if n_req <= n_available:
            print(f"{daily:>13,}  {14:>10}  {mde_try:>14.1%}")
            break

## Real-World Example 2: Underpowered Study Simulation

Key insight: underpowered studies that run until significance are a major source of irreproducible science.

We simulate 2000 experiments with 20% power to show:
- Most true effects are missed (80% false negative rate)
- Of the "significant" findings, a large fraction are false positives from chance

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 2: False discovery rate in underpowered studies
# -----------------------------------------------------------------------

def simulate_experiments(
    true_effect: float,
    sigma: float,
    n_per_group: int,
    n_simulations: int = 2000,
    alpha: float = 0.05,
    prop_h1_true: float = 0.30
) -> dict:
    """
    Simulate a mixture of null (H0) and true-effect (H1) experiments.

    Parameters
    ----------
    true_effect   : the true difference when H1 is true
    prop_h1_true  : prior probability that H1 is true (30% means 30% of tests have real effects)

    Returns
    -------
    dict with TP, FP, TN, FN counts and derived rates
    """
    rng = np.random.default_rng(42)
    tp = fp = tn = fn = 0

    for _ in range(n_simulations):
        # Randomly decide if this experiment has a true effect
        h1_is_true = rng.random() < prop_h1_true

        if h1_is_true:
            mu_diff = true_effect    # true effect exists
        else:
            mu_diff = 0.0            # null is true

        # Generate data
        control = rng.normal(0, sigma, n_per_group)
        treatment = rng.normal(mu_diff, sigma, n_per_group)

        # Two-sample t-test
        t_stat, p_val = stats.ttest_ind(control, treatment)
        reject_h0 = p_val < alpha

        if h1_is_true:
            if reject_h0:
                tp += 1    # true positive (correctly detected)
            else:
                fn += 1    # false negative (missed real effect)
        else:
            if reject_h0:
                fp += 1    # false positive (phantom discovery!)
            else:
                tn += 1    # true negative (correctly retained)

    total = n_simulations
    total_significant = tp + fp
    fdr = fp / total_significant if total_significant > 0 else 0.0
    power_empirical = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    return {
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "power_empirical": power_empirical,
        "fdr": fdr,
        "false_discoveries_pct": fdr * 100
    }


sigma = 10.0
true_delta = 3.0  # effect size d = 0.30 (small-to-medium)

# Underpowered (n=20 per group -- power ~20%)
n_underpower = 20
theoretical_power_low = compute_power(n_underpower, true_delta, sigma)

# Adequately powered (n=90 per group -- power ~80%)
n_adequate = sample_size_for_power(true_delta, sigma, target_power=0.80)
theoretical_power_adeq = compute_power(n_adequate, true_delta, sigma)

results_low = simulate_experiments(true_delta, sigma, n_underpower, prop_h1_true=0.30)
results_adeq = simulate_experiments(true_delta, sigma, n_adequate, prop_h1_true=0.30)

print(f"Simulation: delta={true_delta}, sigma={sigma}, d={true_delta/sigma:.2f}, prop(H1 true)=30%")
print()
print(f"{'Metric':<35}  {'Underpowered (n='+str(n_underpower)+')':>22}  {'Adequate (n='+str(n_adequate)+')':>20}")
print("-" * 82)
print(f"{'Theoretical power':<35}  {theoretical_power_low:>22.1%}  {theoretical_power_adeq:>20.1%}")
print(f"{'Empirical power (TP rate)':<35}  {results_low['power_empirical']:>22.1%}  {results_adeq['power_empirical']:>20.1%}")
print(f"{'False Discovery Rate (FDR)':<35}  {results_low['fdr']:>22.1%}  {results_adeq['fdr']:>20.1%}")
print(f"{'True Positives':<35}  {results_low['tp']:>22d}  {results_adeq['tp']:>20d}")
print(f"{'False Positives':<35}  {results_low['fp']:>22d}  {results_adeq['fp']:>20d}")
print()
print(f"In underpowered study: {results_low['false_discoveries_pct']:.0f}% of significant findings are false!")
print(f"In adequately powered: {results_adeq['false_discoveries_pct']:.0f}% of significant findings are false")

## Real-World Example 3 + Comparison: Power Curves by Effect Size

A power curve family visualizes the required n across effect sizes.
This is the key tool for deciding whether a proposed experiment is feasible.

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 3 + Comparison: Required n for power=0.80 table and
#   power curve family by effect size with matplotlib
# -----------------------------------------------------------------------

# ------ Required n summary table ------
effect_sizes_d_show = [0.1, 0.2, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0]
sigma = 1.0

print("Required n per group for 80% power, alpha=0.05, two-tailed:")
print(f"{'Cohen d':<10}  {'n per group':>12}  {'Total n':>10}  {'Interpretation':>20}")
print("-" * 60)
for d in effect_sizes_d_show:
    n_req = sample_size_for_power(d * sigma, sigma, target_power=0.80)
    interp = {0.1: "trivial", 0.2: "small", 0.3: "small-med",
              0.5: "medium", 0.8: "large", 1.0: "v.large",
              1.5: "huge", 2.0: "massive"}.get(d, "")
    print(f"d={d:<6}    {n_req:>12d}  {2*n_req:>10d}  {interp:>20}")

# ------ Power curve family ------
ns_plot = np.arange(5, 401, 5)
effect_sizes_plot = [0.2, 0.5, 0.8, 1.0]
colors_plot = ["steelblue", "darkorange", "seagreen", "crimson"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: power curves
for d, color in zip(effect_sizes_plot, colors_plot):
    delta = d * sigma
    pows = [compute_power(n, delta, sigma) for n in ns_plot]
    n_80 = sample_size_for_power(delta, sigma, target_power=0.80)
    ax1.plot(ns_plot, pows, color=color, linewidth=2, label=f"d={d} (n80={n_80})")

ax1.axhline(0.80, color="black", linestyle=":", linewidth=2, label="80% target")
ax1.axhline(0.50, color="gray", linestyle=":", linewidth=1, alpha=0.5, label="50% (coin flip)")
ax1.set_xlabel("Sample Size per Group", fontsize=12)
ax1.set_ylabel("Statistical Power", fontsize=12)
ax1.set_title("Power Curves by Effect Size", fontsize=13, fontweight="bold")
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Right: required n bar chart by effect size and power level
power_targets = [0.70, 0.80, 0.90]
x_pos = np.arange(len(effect_sizes_plot))
bar_width = 0.25
bar_colors = ["steelblue", "darkorange", "seagreen"]

for j, (pw, color) in enumerate(zip(power_targets, bar_colors)):
    ns_req = [sample_size_for_power(d * sigma, sigma, target_power=pw) for d in effect_sizes_plot]
    bars = ax2.bar(x_pos + j * bar_width, ns_req, bar_width, label=f"Power={pw:.0%}",
                   color=color, alpha=0.8)
    for bar, n_val in zip(bars, ns_req):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 str(n_val), ha="center", va="bottom", fontsize=8)

ax2.set_xlabel("Effect Size (Cohen's d)", fontsize=12)
ax2.set_ylabel("Required n per Group", fontsize=12)
ax2.set_title("Required n by Effect Size and Power", fontsize=13, fontweight="bold")
ax2.set_xticks(x_pos + bar_width)
ax2.set_xticklabels([f"d={d}" for d in effect_sizes_plot], fontsize=10)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("power_sample_size.png", dpi=100, bbox_inches="tight")
plt.show()

print("
Key insight: detecting small effects (d=0.2) requires ~394 per group.")
print("Quadrupling effect size (d=0.8) reduces n by 15x -- this is the 1/delta^2 relationship.")